# Práctica — Clase 5. Predicción sin modelo: Monte Carlo y Diferencias Temporales

Esta práctica te toca a ti. Trabajas de forma autónoma con el mismo paseo aleatorio que construiste en clase, y resuelves dos ejercicios sobre Monte Carlo (MC) y Diferencias Temporales (TD, del inglés Temporal-Difference).

**Cómo trabajar con este notebook:**
1. Ejecuta primero la sección "Preparación", que reconstruye el entorno de la clase
2. En cada ejercicio, completa las líneas marcadas con `# TODO`
3. Ejecuta tu solución y compara el resultado con lo que se describe en el enunciado
4. Solo después, despliega la solución oculta para comprobar tu respuesta

> **No mires la solución antes de intentarlo.** El valor de este ejercicio está en pelearte un poco con la regla de actualización, no en leer la respuesta.


## Preparación: recupera el paseo aleatorio de la clase

Estas celdas reproducen tal cual el entorno que construiste en el notebook de teoría: la cadena de cinco estados internos, sus valores reales y `generar_episodio`. Los ejercicios de hoy reutilizan estos mismos nombres, así que no los cambies.


In [ ]:
import numpy as np

N_ESTADOS_INTERNOS = 5
ESTADO_INICIAL = 3          # C, el estado central de la cadena
TERMINAL_IZQUIERDA = 0
TERMINAL_DERECHA = N_ESTADOS_INTERNOS + 1

# Valor real de cada estado interno: V(estado i) = i / (N_ESTADOS_INTERNOS + 1)
VALORES_REALES = np.array(
    [i / (N_ESTADOS_INTERNOS + 1) for i in range(1, N_ESTADOS_INTERNOS + 1)]
)


def generar_episodio():
    '''
    Simula un episodio de paseo aleatorio y devuelve la lista de
    (estado, recompensa) que atraviesa, terminando en el estado terminal.
    '''
    estado = ESTADO_INICIAL
    trayectoria = []
    while True:
        # Con probabilidad 0.5 te mueves a la izquierda o a la derecha
        paso = -1 if np.random.rand() < 0.5 else 1
        siguiente_estado = estado + paso
        # Solo el extremo derecho da recompensa; el resto de pasos dan cero
        recompensa = 1.0 if siguiente_estado == TERMINAL_DERECHA else 0.0
        trayectoria.append((estado, recompensa))
        estado = siguiente_estado
        if estado in (TERMINAL_IZQUIERDA, TERMINAL_DERECHA):
            break
    return trayectoria


### Comprobación rápida

Genera un episodio de ejemplo para confirmar que el entorno funciona antes de meterte en los ejercicios.


In [ ]:
np.random.seed(0)
print(generar_episodio())


---

## Ejercicio 1: completa TD(0) desde cero

En clase viste la regla de actualización de TD(0):

```
V(S) ← V(S) + α · [R + γ·V(S') − V(S)]
```

En este ejercicio tienes toda la estructura de `td0_prediccion_ejercicio` ya escrita —el bucle de episodios, el paso por el paseo aleatorio, el cálculo del RMSE—, pero faltan las dos líneas centrales: calcular el error TD y usarlo para actualizar V. Complétalas.

**Pistas:**
- `v_siguiente` ya está calculado para ti: vale 0.0 si el siguiente estado es terminal, y si no, la estimación actual `V[siguiente_estado - 1]`
- El error TD es `recompensa + gamma * v_siguiente - V[idx]`
- La actualización incremental es `V[idx] += alpha * error_td`
- Si tu implementación es correcta, tras 200 episodios con α = 0.1 el RMSE final debería quedar por debajo de 0.1 aproximadamente (el valor exacto depende de la semilla aleatoria)


In [ ]:
def td0_prediccion_ejercicio(n_episodios, alpha, gamma=1.0):
    '''Estima V para los estados internos 1..5 con TD(0).'''
    V = np.full(N_ESTADOS_INTERNOS, 0.5)
    historial_rmse = []

    for _ in range(n_episodios):
        estado = ESTADO_INICIAL
        while estado not in (TERMINAL_IZQUIERDA, TERMINAL_DERECHA):
            paso = -1 if np.random.rand() < 0.5 else 1
            siguiente_estado = estado + paso
            recompensa = 1.0 if siguiente_estado == TERMINAL_DERECHA else 0.0

            if siguiente_estado in (TERMINAL_IZQUIERDA, TERMINAL_DERECHA):
                v_siguiente = 0.0
            else:
                v_siguiente = V[siguiente_estado - 1]

            idx = estado - 1

            # TODO: calcula el error TD: recompensa + gamma * v_siguiente - V[idx]
            error_td = None

            # TODO: actualiza V[idx] sumando alpha * error_td (regla de TD(0))

            estado = siguiente_estado

        rmse = np.sqrt(np.mean((V - VALORES_REALES) ** 2))
        historial_rmse.append(rmse)

    return V, historial_rmse


np.random.seed(0)
V_td_ejercicio, historial_td_ejercicio = td0_prediccion_ejercicio(200, alpha=0.1)
print("V estimado:", V_td_ejercicio)
print("Valores reales:", VALORES_REALES)
print(f"RMSE final: {historial_td_ejercicio[-1]:.4f}")


<details>
<summary>💡 Ver solución</summary>

```python
def td0_prediccion_ejercicio(n_episodios, alpha, gamma=1.0):
    """Estima V para los estados internos 1..5 con TD(0)."""
    V = np.full(N_ESTADOS_INTERNOS, 0.5)
    historial_rmse = []

    for _ in range(n_episodios):
        estado = ESTADO_INICIAL
        while estado not in (TERMINAL_IZQUIERDA, TERMINAL_DERECHA):
            paso = -1 if np.random.rand() < 0.5 else 1
            siguiente_estado = estado + paso
            recompensa = 1.0 if siguiente_estado == TERMINAL_DERECHA else 0.0

            if siguiente_estado in (TERMINAL_IZQUIERDA, TERMINAL_DERECHA):
                v_siguiente = 0.0
            else:
                v_siguiente = V[siguiente_estado - 1]

            idx = estado - 1

            # Error TD: lo que acabas de observar (recompensa + valor
            # descontado del siguiente estado) menos lo que ya creías
            error_td = recompensa + gamma * v_siguiente - V[idx]

            # Actualización incremental hacia ese error, escalada por alpha
            V[idx] += alpha * error_td

            estado = siguiente_estado

        rmse = np.sqrt(np.mean((V - VALORES_REALES) ** 2))
        historial_rmse.append(rmse)

    return V, historial_rmse


np.random.seed(0)
V_td_ejercicio, historial_td_ejercicio = td0_prediccion_ejercicio(200, alpha=0.1)
print("V estimado:", V_td_ejercicio)
print("Valores reales:", VALORES_REALES)
print(f"RMSE final: {historial_td_ejercicio[-1]:.4f}")

# Salida esperada aproximada (con la semilla 0):
# V estimado: [0.1107 0.3371 0.4691 0.6996 0.8678]
# Valores reales: [0.1667 0.3333 0.5    0.6667 0.8333]
# RMSE final: 0.0357
```
</details>


---

## Ejercicio 2: completa Monte Carlo de primera visita y compara ambos métodos

Ahora te toca MC. La función `monte_carlo_primera_visita_ejercicio` recorre cada episodio de atrás hacia adelante para calcular el retorno G de cada paso, y se queda con el retorno de la primera vez que aparece cada estado. Completa el cálculo del retorno y la actualización de V.

Después, con `td0_prediccion_ejercicio` del ejercicio anterior ya completo, vas a comparar el error final de MC y TD(0) con tres tasas de aprendizaje distintas: α = 0.01, α = 0.05 y α = 0.15.

**Pistas:**
- El retorno se acumula recorriendo la trayectoria al revés: `G = recompensa + gamma * G`
- Como recorres la trayectoria en orden inverso, la última asignación que hagas sobre `retornos_por_estado[estado]` corresponde a la primera visita de ese estado en el episodio: guarda ahí el G que acabas de calcular
- La actualización de V usa la misma regla incremental que en TD, pero apunta al retorno completo: `V[idx] += alpha * (G - V[idx])`
- En la comparación final, espera ver que un α más alto da un RMSE más alto en ambos métodos: es el ruido residual del que hablasteis en clase


In [ ]:
def monte_carlo_primera_visita_ejercicio(n_episodios, alpha, gamma=1.0):
    '''Estima V para los estados internos 1..5 con MC de primera visita.'''
    V = np.full(N_ESTADOS_INTERNOS, 0.5)
    historial_rmse = []

    for _ in range(n_episodios):
        trayectoria = generar_episodio()

        G = 0.0
        retornos_por_estado = {}
        for estado, recompensa in reversed(trayectoria):
            # TODO: acumula el retorno: G = recompensa + gamma * G
            G = None

            # TODO: guarda G como el retorno de "estado" en retornos_por_estado
            #       (recuerda: la última asignación sobre un estado repetido
            #       es la que corresponde a su primera visita)

        for estado, G in retornos_por_estado.items():
            if estado not in (TERMINAL_IZQUIERDA, TERMINAL_DERECHA):
                idx = estado - 1
                # TODO: actualiza V[idx] hacia G con la regla incremental
                #       V[idx] += alpha * (G - V[idx])
                pass

        rmse = np.sqrt(np.mean((V - VALORES_REALES) ** 2))
        historial_rmse.append(rmse)

    return V, historial_rmse


np.random.seed(0)
V_mc_ejercicio, historial_mc_ejercicio = monte_carlo_primera_visita_ejercicio(200, alpha=0.1)
print("V estimado:", V_mc_ejercicio)
print("Valores reales:", VALORES_REALES)
print(f"RMSE final: {historial_mc_ejercicio[-1]:.4f}")


<details>
<summary>💡 Ver solución</summary>

```python
def monte_carlo_primera_visita_ejercicio(n_episodios, alpha, gamma=1.0):
    """Estima V para los estados internos 1..5 con MC de primera visita."""
    V = np.full(N_ESTADOS_INTERNOS, 0.5)
    historial_rmse = []

    for _ in range(n_episodios):
        trayectoria = generar_episodio()

        G = 0.0
        retornos_por_estado = {}
        for estado, recompensa in reversed(trayectoria):
            # Acumula el retorno hacia atrás desde el final del episodio
            G = recompensa + gamma * G
            # Al recorrer la trayectoria al revés, la última asignación
            # para un estado repetido es la de su primera visita
            retornos_por_estado[estado] = G

        for estado, G in retornos_por_estado.items():
            if estado not in (TERMINAL_IZQUIERDA, TERMINAL_DERECHA):
                idx = estado - 1
                # Actualización incremental hacia el retorno G
                V[idx] += alpha * (G - V[idx])

        rmse = np.sqrt(np.mean((V - VALORES_REALES) ** 2))
        historial_rmse.append(rmse)

    return V, historial_rmse


np.random.seed(0)
V_mc_ejercicio, historial_mc_ejercicio = monte_carlo_primera_visita_ejercicio(200, alpha=0.1)
print("V estimado:", V_mc_ejercicio)
print("Valores reales:", VALORES_REALES)
print(f"RMSE final: {historial_mc_ejercicio[-1]:.4f}")

# Salida esperada aproximada (con la semilla 0):
# V estimado: [0.1927 0.2591 0.3563 0.6763 0.8233]
# Valores reales: [0.1667 0.3333 0.5    0.6667 0.8333]
# RMSE final: 0.0735
```
</details>


### Ahora compara: MC vs TD(0) con distintas tasas de aprendizaje

Con las dos funciones ya completas, calcula el RMSE final —promediado sobre varias ejecuciones, para suavizar el ruido de una sola tirada— de MC y TD(0) para α = 0.01, α = 0.05 y α = 0.15.

**Pistas:**
- Usa `rmse_promedio` para repetir el experimento con varias semillas y quedarte con el promedio del historial de RMSE
- Llama a `monte_carlo_primera_visita_ejercicio` y a `td0_prediccion_ejercicio` con cada α, sobre 300 episodios
- El último valor del historial promediado es el RMSE final que va en la tabla


In [ ]:
def rmse_promedio(funcion_prediccion, n_episodios, alpha, n_ejecuciones=20):
    '''
    Repite el experimento n_ejecuciones veces con semillas distintas y
    promedia el historial de RMSE, para suavizar el ruido de una sola
    ejecución antes de comparar tasas de aprendizaje.
    '''
    acumulado = np.zeros(n_episodios)
    for ejecucion in range(n_ejecuciones):
        np.random.seed(ejecucion)
        _, historial = funcion_prediccion(n_episodios, alpha)
        acumulado += np.array(historial)
    return acumulado / n_ejecuciones


ALPHAS_A_PROBAR = [0.01, 0.05, 0.15]
N_EPISODIOS_COMPARACION = 300

print(f"{'α':>6} | {'RMSE MC':>10} | {'RMSE TD(0)':>10}")
for alpha in ALPHAS_A_PROBAR:
    # TODO: usa rmse_promedio con monte_carlo_primera_visita_ejercicio y
    #       quédate con el último valor del historial promediado
    rmse_mc_final = None

    # TODO: haz lo mismo para TD(0) con td0_prediccion_ejercicio
    rmse_td_final = None

    print(f"{alpha:>6} | {rmse_mc_final:>10.4f} | {rmse_td_final:>10.4f}")


<details>
<summary>💡 Ver solución</summary>

```python
def rmse_promedio(funcion_prediccion, n_episodios, alpha, n_ejecuciones=20):
    """
    Repite el experimento n_ejecuciones veces con semillas distintas y
    promedia el historial de RMSE, para suavizar el ruido de una sola
    ejecución antes de comparar tasas de aprendizaje.
    """
    acumulado = np.zeros(n_episodios)
    for ejecucion in range(n_ejecuciones):
        np.random.seed(ejecucion)
        _, historial = funcion_prediccion(n_episodios, alpha)
        acumulado += np.array(historial)
    return acumulado / n_ejecuciones


ALPHAS_A_PROBAR = [0.01, 0.05, 0.15]
N_EPISODIOS_COMPARACION = 300

print(f"{'α':>6} | {'RMSE MC':>10} | {'RMSE TD(0)':>10}")
for alpha in ALPHAS_A_PROBAR:
    rmse_mc_final = rmse_promedio(
        monte_carlo_primera_visita_ejercicio, N_EPISODIOS_COMPARACION, alpha
    )[-1]
    rmse_td_final = rmse_promedio(
        td0_prediccion_ejercicio, N_EPISODIOS_COMPARACION, alpha
    )[-1]
    print(f"{alpha:>6} | {rmse_mc_final:>10.4f} | {rmse_td_final:>10.4f}")

# Salida esperada aproximada:
#      α |    RMSE MC | RMSE TD(0)
#   0.01 |     0.0427 |     0.0373
#   0.05 |     0.0571 |     0.0348
#   0.15 |     0.1039 |     0.0640
#
# Fíjate en el patrón: para cada α, TD(0) queda por debajo de MC (menor
# varianza), y para ambos métodos el error final crece con α (más ruido
# residual con una tasa de aprendizaje más alta). Es el balance entre
# sesgo y varianza de la clase, ahora medido en números.
```
</details>


---

## Para seguir practicando

- **Añade un gráfico**: usa `matplotlib` para dibujar, en un mismo eje, el RMSE frente a episodios de MC y TD(0) con tu α favorita, y comprueba visualmente lo que acabas de ver en la tabla
- **Prueba cada visita**: modifica tu `monte_carlo_primera_visita_ejercicio` para que actualice V en cada ocurrencia del estado dentro del episodio, no solo en la primera, y comprueba que el resultado final apenas cambia

**Siguiente parada:** en la Clase 6 vas a reutilizar esta misma idea de Diferencias Temporales para pasar de evaluar una política a mejorarla, con dos algoritmos de control —SARSA y Q-learning— sobre el entorno de Cliff Walking. ¡Guarda este notebook, la intuición de "ajustar con lo que acabas de ver" vuelve a aparecer!
